# Exercise 9: series

* pandas series vs numpy arrays [explanation](https://jakevdp.github.io/PythonDataScienceHandbook/03.01-introducing-pandas-objects.html)

### Common series operations
These are the most common series operations we use. Refer to the `pandas` docs for even more!

* Getting dates, hours, minutes from datetime types (`df.datetime_col.dt.date`)
* Parsing strings (`df.string_col.str.split('_')`)

### Common geoseries operations
These are the most common. Refer to the `geopandas` docs for even more!

* `distance` between 2 points or a point to a polygon or line [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.distance.html)
* `intersects`: [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.intersects.html)
* `within`: [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.within.html)
* `contains`: [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.contains.html)

In fact, we've often used geoseries methods without even realizing it. Often, we'd create a new column that stores either the line's length or a polygon's area. `gdf.geometry` is a geoseries, and we call methods on that geoseries, and add that as a new column.

For calculations like `length`, `area`, and `distance`, we need to use a projected CRS that has units like meters or feet. We cannot use decimal degrees (do not use WGS 84 / EPSG:3326)! Distance calculations must be done only once the spherical 3D Earth has been converted into a 2D plane.

* `length`: get the length of a line (`gdf.geometry.length`)
* `area`: get the area of a polygon (`gdf.geometry.area`)
* `centroid`: get the centroid of a polygon (`gdf.geometry.centroid`)
* `x`: get the x coordinate of a point (`gdf.geometry.x`)
* `y`: get the y coordinate of a point (`gdf.geometry.y`)

### Arrays
* Occasionally, we may even use arrays, especially when the datasets get even larger but we have simple mathematical calculations
* If we need to apply an exponential decay function to a distance column, we essentially want to multiple `distance` by some number
* Since this exponential decay function is somewhat custom and requires us to write our own formula, we would extract the column as a series (`df.distance`) and multiply each value by some other number.
* Even quicker is to use `numpy` with `distance_array = np.array(df.distance)` and get `exponential_array = distance_array*some_number`

In [1]:
import geopandas as gpd
import intake
import numpy as np
import pandas as pd

catalog = intake.open_catalog(
    "../_shared_utils/shared_utils/shared_data_catalog.yml")

/opt/conda/lib/python3.9/site-packages/geopandas/_compat.py:123: UserWarning: The Shapely GEOS version (3.11.1-CAPI-1.17.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  warnings.warn(
/tmp/ipykernel_535/3598373760.py:1: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_pygeos.html).
  im

In [2]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

If you're asking how far is a transit stop from the interstate, you'd want the distance of every point (every row) compared to an interstate highway geometry.

Let's prep the datasets to use series / geoseries to do this.

In [3]:
#stops = catalog.ca_transit_stops.read()[["agency", "stop_id", 
                                         #"stop_name", "geometry"]]
highways = catalog.state_highway_network.read()

In [4]:
#stops.shape, type(stops)

In [5]:
#stops.sample()

In [6]:
#highways.shape, type(highways)

Since we want to know the distance from a stop's point to the interstate generally, we need a dissolve. We don't want to compare the distance against the I-5, the I-10 individually, but to the interstate system as a whole.

In [7]:
interstates = (highways[highways.RouteType=="Interstate"]
               .dissolve()
               .reset_index()
               [["geometry"]]
              )

In [8]:
interstates.shape

(1, 1)

In [9]:
# This is still a gdf, just with 1 column
type(interstates)

geopandas.geodataframe.GeoDataFrame

In [10]:
# Pulling out the individual column, it becomes a series/geoseries.
# It's a geoseries here because we had a gdf. 
# If it was a df, it would be a series.
#print(type(stops.geometry))
#print(type(interstates.geometry))

Distance is something you can calculate using `geopandas`.

Specifically, it takes a geoseries on the left, and either a geoseries or a single geometry on the right.

An example of having 2 geoseries would be comparing the distance between 2 points. On the left, it would be a geoseries of the origin points and on the right, destination points.

In [11]:
# We get a warning if we leave it in EPSG:4326!
# stops.geometry.distance(interstates.geometry.iloc[0])

In [12]:
# stops_geom = stops.to_crs("EPSG:2229").geometry

In [13]:
# type(stops_geom)

In [14]:
# stops_geom.head()

In [15]:
# interstates_geom = interstates.to_crs("EPSG:2229").geometry.iloc[0]
# distance_series = stops_geom.distance(interstates_geom)

In [16]:
# distance_series = stops_geom.distance(interstates_geom)

In [17]:
# distance_series.head()

In [18]:
# Let's make sure that for every stop, a distance is calculated
# print(f"# rows in stops: {len(stops_geom)}")
# print(f"# rows in stops: {len(distance_series)}")

In [19]:
# distance is numeric, not a geometry, so we're back to being a series
# type(distance_series)

What can we do with this? 

We usually add it as a new column. Since we did nothing to shift the index, we can just attach the series back to our gdf.

Getting a distance calculation using geoseries is much quicker than a row-wise lambda function where you calculate the distance.

```
Alternative method that's slower:
      
interstate_geom = interstates.geometry.iloc[0]

stops = stops.assign(
   distance = stops.geometry.apply(
         lambda x: x.distance(interstate_geom))
)   
```

In [20]:
#stops = stops.assign(
#    distance_to_interstate = distance_series
#)

In [21]:
#%%timeit
#distance_series = stops_geom.distance(interstates_geom)

In [22]:
"""
%%timeit
stops.assign(
   distance = stops.geometry.apply(
         lambda x: x.distance(interstates_geom))
)  """

'\n%%timeit\nstops.assign(\n   distance = stops.geometry.apply(\n         lambda x: x.distance(interstates_geom))\n)  '

In [23]:
"""
import dask_geopandas as dg

stops_gddf = dg.from_geopandas(stops, npartitions=2)
stops_geom_dg = stops_gddf.to_crs("EPSG:2229").geometry """

'\nimport dask_geopandas as dg\n\nstops_gddf = dg.from_geopandas(stops, npartitions=2)\nstops_geom_dg = stops_gddf.to_crs("EPSG:2229").geometry '

In [24]:
 """ %%timeit

distance_series = stops_geom_dg.distance(interstates_geom)"""

' %%timeit\n\ndistance_series = stops_geom_dg.distance(interstates_geom)'

## To Do
* Use the `stop_times` table and `stops` table.
* Calculate the straight line distance between the first and last stop for each trip. Call this column `trip_distance`
* Calculate the distance between each stop to the nearest interstate. For each trip, keep the value for the stop that's the closest to the interstate. Call this column `shortest_distance_hwy`.
* For each trip, add these 2 new columns, but use series, geoseries, and/or arrays to assign it.
* Provide a preview of the resulting df (do not export)

In [25]:
GCS_FILE_PATH = ("gs://calitp-analytics-data/data-analyses/"
                 "rt_delay/compiled_cached_views/"
                )

analysis_date = "2023-01-18"
STOP_TIMES_FILE = f"{GCS_FILE_PATH}st_{analysis_date}.parquet"
STOPS_FILE = f"{GCS_FILE_PATH}stops_{analysis_date}.parquet"

highways = catalog.state_highway_network.read()

### Prep

#### Prep Stops

In [28]:
def prep_stops():
    """
    Returns all the stops
    on a particular day.
    """
    df = gpd.read_parquet(STOPS_FILE)
    df = df[['feed_key', 'stop_id', 'geometry']]
    
    # Change to feet
    df = df.to_crs("EPSG:2229")
    
    return df

In [29]:
all_stops = prep_stops()

In [30]:
all_stops.stop_id.nunique()

51514

In [31]:
all_stops.shape

(84688, 3)

### Calculate the straight line distance between the first and last stop for each trip. Call this column trip_distance

In [32]:
def prep_stoptimes():
    df = pd.read_parquet(STOP_TIMES_FILE)
    
    # Only keep sequence
    df = df[['feed_key','trip_id','stop_id','stop_sequence']]
    
    # Sort so sequence goes from lowest to highest value
    df = df.sort_values(by = ['feed_key','trip_id','stop_sequence']).reset_index(drop = True)
    return df 

In [33]:
stoptimes = prep_stoptimes()

In [34]:
stoptimes.stop_id.nunique()

49560

In [35]:
stoptimes.shape

(3589931, 4)

In [36]:
#_merge results
#both          3589718
#left_only        3670
#right_only        263

# pd.merge(stops, stop_times, how = 'outer', on = ['feed_key', 'stop_id'], indicator = True)[['_merge']].value_counts()

In [37]:
def complete_stop_info():
    stop_times = prep_stoptimes()
    stops = prep_stops()
    
    m1 = pd.merge(stops, stop_times, how = 'inner', on = ['feed_key', 'stop_id'])
    
    return m1

In [38]:
stop_w_times = complete_stop_info()

In [81]:
stop_w_times.crs

<Projected CRS: EPSG:2229>
Name: NAD83 / California zone 5 (ftUS)
Axis Info [cartesian]:
- X[east]: Easting (US survey foot)
- Y[north]: Northing (US survey foot)
Area of Use:
- name: United States (USA) - California - counties Kern; Los Angeles; San Bernardino; San Luis Obispo; Santa Barbara; Ventura.
- bounds: (-121.42, 32.76, -114.12, 35.81)
Coordinate Operation:
- name: SPCS83 California zone 5 (US Survey feet)
- method: Lambert Conic Conformal (2SP)
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [39]:
# Trip IDS are slightly different
# stops_m1.head(10)

In [40]:
def first_last_trip():
    """
    Keep only the first and last stop.
    """
    m1 = complete_stop_info()
    
    sort_values = ['feed_key','trip_id','stop_sequence']
    duplicate_values = ['feed_key','trip_id',]
    
    # Find the FIRST stop of every trip
    first = (m1
                  .sort_values(by = sort_values)
                  .drop_duplicates(subset = duplicate_values)
                  .reset_index(drop = True)
                 )
    # Find the LAST stop of every trip
    last = (m1
                 .sort_values(by = sort_values,
                              ascending = [True, True, False])
                 .drop_duplicates(subset = duplicate_values)
                 .reset_index(drop = True)
                )
    return first,last

In [41]:
first_stop, last_stop = first_last_trip()

In [42]:
len(first_stop), first_stop.stop_id.nunique()

(109116, 3405)

In [43]:
# strange that the first sequence is not necessarily 1...
first_stop.stop_sequence.value_counts().head()

1     94213
0     13073
2      1297
10      114
25      105
Name: stop_sequence, dtype: int64

In [78]:
# strange that the first sequence is not necessarily 1...
last_stop.stop_sequence.value_counts().head()

3     9623
26    2361
23    2348
21    2255
24    2223
Name: stop_sequence, dtype: int64

In [80]:
last_stop.crs == first_stop.crs, first_stop.crs

(True,
 <Projected CRS: EPSG:2229>
 Name: NAD83 / California zone 5 (ftUS)
 Axis Info [cartesian]:
 - X[east]: Easting (US survey foot)
 - Y[north]: Northing (US survey foot)
 Area of Use:
 - name: United States (USA) - California - counties Kern; Los Angeles; San Bernardino; San Luis Obispo; Santa Barbara; Ventura.
 - bounds: (-121.42, 32.76, -114.12, 35.81)
 Coordinate Operation:
 - name: SPCS83 California zone 5 (US Survey feet)
 - method: Lambert Conic Conformal (2SP)
 Datum: North American Datum 1983
 - Ellipsoid: GRS 1980
 - Prime Meridian: Greenwich)

In [96]:
# first_stop.loc[first_stop.trip_id == "DX12S1213113"]

In [97]:
# last_stop.loc[last_stop.trip_id == "DX12S1213113"]

In [98]:
# Check out the stop sequences that aren't 1 for one trip...
# stop_w_times.loc[stop_w_times.trip_id == "DX12S1213113"]

* Attach distance last_first back to dataframe

In [85]:
def find_distance():
    """
    Find distance between the first and last stop
    for each trip
    """
    first_stop, last_stop = first_last_trip()
    
    # All trips plus stop times
    m1 = complete_stop_info()
    
    # Turn the last and first stop into a geo series
    first_stop_series = first_stop.geometry
    last_stop_series = last_stop.geometry
    
    # Find distance of last stop to first
    trip_distance = last_stop_series.distance(first_stop_series)
    
    # Attach the trip_distance series to firststop in miles
    first_stop = first_stop.assign(
    distance_last_first_stop = trip_distance/5_280)
    
    first_stop = first_stop[['feed_key', 'stop_id','trip_id','distance_last_first_stop']]
    
    # Merge first stop back to m1.
    m2 = pd.merge(m1, first_stop, on = ['feed_key','trip_id'], how = 'inner')
    
    m2 = (m2.drop(columns = ['stop_id_x','geometry','stop_sequence'])
          .rename(columns = {'stop_id_y':'stop_id'})
         )
        
    return m2

In [86]:
trips_w_dist = find_distance()

In [48]:
trips_w_dist.shape

(109116, 4)

In [50]:
stop_w_times.shape

(3589718, 5)

In [51]:
# Make sure all trips are still there
stop_w_times_tripid = set(stop_w_times.trip_id.unique().tolist())
trips_w_dist_trip_id = set(trips_w_dist.trip_id.unique().tolist())
stop_w_times_tripid - trips_w_dist_trip_id

set()

In [92]:
# Attach distance last to first stop by trip_id and feed_key
# pd.merge(trips_w_dist, stop_w_times, on = ['feed_key','trip_id'], how = 'outer', indicator = True)[['_merge']].value_counts()

In [53]:
# m1 = pd.merge(trips_w_dist, stop_w_times, on = ['feed_key','trip_id'], how = 'inner')

In [54]:
# m1 = m1.drop(columns = ['stop_id_x','geometry','stop_sequence'])

In [55]:
# m1 = m1.rename(columns = {'stop_id_y':'stop_id'})

In [91]:
# m1.head(3)

### Calculate the distance between each stop to the nearest interstate. For each trip, keep the value for the stop that's the closest to the interstate

In [94]:
def find_interstate_distance(interstate_gdf):
    interstates_geom = interstate_gdf.to_crs("EPSG:2229").geometry.iloc[0]
    
    # Find all stops and trip times.
    m1 = prep_stops()
    
    # Geoseries
    m1_series = m1.geometry
    
    # Find distance of all the stops to interstate geom
    distance_series = m1_series.distance(interstates_geom)
    
    # Attach distance
    m1 = m1.assign(
        distance_stop_interstate = distance_series/5_280)
    
    return m1

In [95]:
all_stops = find_interstate_distance(interstates)

### Combine 

In [105]:
def interstate_last_first_stop(interstate_gdf):
    
    # Get distance from last to first stop
    trips_w_dist = find_distance()
    
    # Drop duplicates
    trips_w_dist = trips_w_dist.drop_duplicates().reset_index(drop = True) 
    
    # Get distance from interstate
    interstate_dist = find_interstate_distance(interstate_gdf)
    
    # Merge
    m1 = pd.merge(trips_w_dist,interstate_dist, on = ['feed_key', 'stop_id'], how = 'inner')
    
    # Keep only the nearest stop to interstate for each trip 
    # So the distance with the smallest value
    m1 = (m1
      .sort_values(['feed_key','trip_id','distance_stop_interstate'])
      .drop_duplicates(subset = ['feed_key','trip_id'])
      .reset_index(drop = True)
     )
    
    m1 = m1.rename(columns = {'distance_stop_interstate':'shortest_distance_hwy'})
    
    return m1

In [106]:
final_df = interstate_last_first_stop(interstates)

In [102]:
#_merge    
#both          3568059
#left_only        3670
#right_only          0
#dtype: int64
#pd.merge(all_stops, m2, on = ['feed_key', 'stop_id'], how = 'outer',indicator = True)[['_merge']].value_counts()

In [107]:
# Check against original merge before all this manipulation
final_df.trip_id.nunique(), stop_w_times.trip_id.nunique()

(100876, 100876)

In [108]:
m3_tripid = set(m3.trip_id.unique().tolist())
stop_w_times_tripid - m3_tripid

set()

In [113]:
final_df.shape

(109116, 6)

In [112]:
final_df.drop_duplicates(['feed_key','stop_id']).head(20)

,feed_key,trip_id,stop_id,distance_last_first_stop,geometry,shortest_distance_hwy
0,008d5112a7e531d0562d26e34d77869d,1080037,8908,8.65,POINT (5557749.878 3539387.307),1.48
14,008d5112a7e531d0562d26e34d77869d,1080051,8900,9.02,POINT (5557830.503 3541368.217),1.63
23,008d5112a7e531d0562d26e34d77869d,1080060,2425,8.65,POINT (5562810.955 3493986.355),1.30
24,008d5112a7e531d0562d26e34d77869d,1080061,327,6.53,POINT (5559677.240 3506947.475),0.45
46,008d5112a7e531d0562d26e34d77869d,1080083,5348,0.84,POINT (5556702.078 3537093.938),1.14
48,008d5112a7e531d0562d26e34d77869d,1080343,1334,9.30,POINT (5548402.224 3536244.904),0.37
68,008d5112a7e531d0562d26e34d77869d,1080363,1071,9.30,POINT (5593957.060 3517918.364),2.27
88,008d5112a7e531d0562d26e34d77869d,1080451,9800,4.82,POINT (5593400.312 3530779.450),0.02
116,008d5112a7e531d0562d26e34d77869d,1080479,9807,4.82,POINT (5571809.792 3517314.644),2.41
144,008d5112a7e531d0562d26e34d77869d,1080591,3125,8.30,POINT (5591663.604 3556357.708),3.73
